In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Dataset paths
DATA_DIR = Path("data")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

print("Training shape:", train.shape)
print("Testing shape:", test.shape)
print("Submission shape:", sample_submission.shape)

display(train.head())


Training shape: (630000, 21)
Testing shape: (270000, 20)
Submission shape: (270000, 2)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [2]:

# Check feature consistency
target = "Irrigation_Need"

train_features = set(train.columns) - {target, "id"}
test_features = set(test.columns) - {"id"}

print("Features match:", train_features == test_features)

print("\nMissing values:")
print(train.isnull().sum().sum())

print("\nTarget distribution:")
print(train[target].value_counts())

print("\nSubmission columns:")
print(sample_submission.columns.tolist())


Features match: True

Missing values:
0

Target distribution:
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64

Submission columns:
['id', 'Irrigation_Need']


In [3]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = train.drop(columns=["id", "Irrigation_Need"])
y = train["Irrigation_Need"]

# Identify categorical columns
cat_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Stratified train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

print("\nCategorical features:")
print(cat_features)

print("\nTraining target distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nValidation target distribution (%):")
print((y_val.value_counts(normalize=True) * 100).round(2))


C:\Users\Akash Kumar\AppData\Local\Temp\ipykernel_612\1725942437.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(


Training shape: (504000, 19)
Validation shape: (126000, 19)

Categorical features:
['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

Training target distribution (%):
Irrigation_Need
Low       58.72
Medium    37.95
High       3.33
Name: proportion, dtype: float64

Validation target distribution (%):
Irrigation_Need
Low       58.72
Medium    37.95
High       3.33
Name: proportion, dtype: float64


In [4]:
from catboost import CatBoostClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)
import time

# Initialize CatBoost
catboost_model = CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    thread_count=-1,
    verbose=100
)

# Train model
start_time = time.time()

catboost_model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50
)

training_time = time.time() - start_time

# Validation predictions
y_pred = catboost_model.predict(X_val).ravel()

# Evaluation
balanced_acc = balanced_accuracy_score(y_val, y_pred)

print("\nTraining Time:", round(training_time, 2), "seconds")
print("Balanced Accuracy:", round(balanced_acc, 6))

print("\nClassification Report:")
print(classification_report(y_val, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_val,
    y_pred,
    labels=["Low", "Medium", "High"]
))


0:	learn: 1.0066278	test: 1.0065631	best: 1.0065631 (0)	total: 422ms	remaining: 4m 12s
100:	learn: 0.0685127	test: 0.0666591	best: 0.0666591 (100)	total: 31.1s	remaining: 2m 33s
200:	learn: 0.0636127	test: 0.0620342	best: 0.0620342 (200)	total: 1m 4s	remaining: 2m 8s
300:	learn: 0.0623376	test: 0.0611180	best: 0.0611179 (299)	total: 1m 38s	remaining: 1m 38s
400:	learn: 0.0613800	test: 0.0604633	best: 0.0604633 (400)	total: 2m 39s	remaining: 1m 19s
500:	learn: 0.0606026	test: 0.0600014	best: 0.0600014 (500)	total: 3m 16s	remaining: 38.8s
599:	learn: 0.0599701	test: 0.0596416	best: 0.0596416 (599)	total: 3m 50s	remaining: 0us

bestTest = 0.05964164567
bestIteration = 599


Training Time: 232.61 seconds
Balanced Accuracy: 0.963828

Classification Report:
              precision    recall  f1-score   support

        High       0.97      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.98      0.98     47815

    accuracy        

In [5]:
from sklearn.metrics import balanced_accuracy_score

# Get predicted probabilities
probs = catboost_model.predict_proba(X_val)
classes = catboost_model.classes_

print("Class order:", classes)

# Search class probability multipliers
results = []

for high_weight in np.arange(1.0, 2.51, 0.1):
    for medium_weight in np.arange(1.0, 1.51, 0.1):

        weights = np.array([
            high_weight if c == "High"
            else medium_weight if c == "Medium"
            else 1.0
            for c in classes
        ])

        adjusted_probs = probs * weights
        predictions = classes[np.argmax(adjusted_probs, axis=1)]

        score = balanced_accuracy_score(y_val, predictions)

        results.append({
            "high_weight": round(high_weight, 2),
            "medium_weight": round(medium_weight, 2),
            "balanced_accuracy": score
        })

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        "balanced_accuracy",
        ascending=False
    ).head(10)
)


Class order: ['High' 'Low' 'Medium']


,high_weight,medium_weight,balanced_accuracy
90,2.5,1.0,0.966655
72,2.2,1.0,0.966573
85,2.4,1.1,0.966569
91,2.5,1.1,0.966555
78,2.3,1.0,0.966538
84,2.4,1.0,0.966524
92,2.5,1.2,0.966474
79,2.3,1.1,0.966438
66,2.1,1.0,0.966435
93,2.5,1.3,0.966348


In [6]:
from sklearn.metrics import balanced_accuracy_score
import numpy as np
import pandas as pd

results = []

for high_weight in np.arange(2.0, 5.01, 0.05):

    weights = np.array([
        high_weight if c == "High" else 1.0
        for c in classes
    ])

    adjusted_probs = probs * weights

    predictions = classes[
        np.argmax(adjusted_probs, axis=1)
    ]

    score = balanced_accuracy_score(y_val, predictions)

    results.append({
        "high_weight": round(high_weight, 2),
        "balanced_accuracy": score
    })

tuning_results = pd.DataFrame(results)

# Display best configurations
display(
    tuning_results.sort_values(
        "balanced_accuracy",
        ascending=False
    ).head(10)
)

best_row = tuning_results.loc[
    tuning_results["balanced_accuracy"].idxmax()
]

print("\nBest High Weight:", best_row["high_weight"])
print("Best Balanced Accuracy:",
      round(best_row["balanced_accuracy"], 6))


,high_weight,balanced_accuracy
57,4.85,0.968256
58,4.90,0.968221
55,4.75,0.968211
59,4.95,0.968200
56,4.80,0.968191
60,5.00,0.968186
53,4.65,0.968146
54,4.70,0.968139
52,4.60,0.968081
50,4.50,0.967929



Best High Weight: 4.85
Best Balanced Accuracy: 0.968256



## Submission 1 — CatBoost with Probability Adjustment

Generate test predictions using CatBoost and a High-class probability multiplier of 4.85.


In [7]:
from pathlib import Path
import numpy as np

# Prepare test features
X_test = test.drop(columns=["id"])

# Predict probabilities on test data
test_probs = catboost_model.predict_proba(X_test)

# Apply optimized class multiplier
best_high_weight = 4.85

weights = np.array([
    best_high_weight if c == "High" else 1.0
    for c in catboost_model.classes_
])

adjusted_probs = test_probs * weights

test_predictions = catboost_model.classes_[
    np.argmax(adjusted_probs, axis=1)
]

# Prepare submission
submission = sample_submission.copy()
submission["Irrigation_Need"] = test_predictions

# Verify submission format
assert len(submission) == len(test)
assert submission["id"].equals(test["id"])
assert submission["Irrigation_Need"].isin(
    ["Low", "Medium", "High"]
).all()

# Save submission
Path("submissions").mkdir(exist_ok=True)

submission.to_csv(
    "submissions/submission_01_catboost.csv",
    index=False
)

print("Submission created successfully!")
print("\nPrediction distribution:")
print(submission["Irrigation_Need"].value_counts())

display(submission.head())


Submission created successfully!

Prediction distribution:
Irrigation_Need
Low       159692
Medium    101261
High        9047
Name: count, dtype: int64


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low



## Model 2 — LightGBM

Train a LightGBM multiclass classifier with categorical features and early stopping. Evaluate balanced accuracy and compare against CatBoost.


In [8]:
import lightgbm as lgb
import pandas as pd
import time

from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report
)

# Convert categorical columns for LightGBM
X_train_lgb = X_train.copy()
X_val_lgb = X_val.copy()

for col in cat_features:
    categories = pd.Categorical(
        pd.concat([X_train[col], X_val[col]])
    ).categories

    X_train_lgb[col] = pd.Categorical(
        X_train[col], categories=categories
    )
    X_val_lgb[col] = pd.Categorical(
        X_val[col], categories=categories
    )

# Initialize LightGBM
lgb_model = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=40,
    colsample_bytree=0.9,
    subsample=0.8,
    subsample_freq=1,
    reg_lambda=3.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

start = time.time()

lgb_model.fit(
    X_train_lgb,
    y_train,
    eval_set=[(X_val_lgb, y_val)],
    eval_metric="multi_logloss",
    categorical_feature=cat_features,
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(200)
    ]
)

print("\nTraining time:", round(time.time() - start, 2), "seconds")

# Predict validation probabilities
lgb_probs = lgb_model.predict_proba(X_val_lgb)

# Baseline predictions
lgb_predictions = lgb_model.classes_[
    lgb_probs.argmax(axis=1)
]

print("\nLightGBM Balanced Accuracy:",
      round(balanced_accuracy_score(y_val, lgb_predictions), 6))

print("\nClassification Report:")
print(classification_report(y_val, lgb_predictions))


c:\Users\Akash Kumar\Desktop\AIS-Competition-2026\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[200]	valid_0's multi_logloss: 0.0595574
[400]	valid_0's multi_logloss: 0.0562715
[600]	valid_0's multi_logloss: 0.055457
[800]	valid_0's multi_logloss: 0.0549405
[1000]	valid_0's multi_logloss: 0.0546308
[1200]	valid_0's multi_logloss: 0.0544619
[1400]	valid_0's multi_logloss: 0.0544208
Did not meet early stopping. Best iteration is:
[1495]	valid_0's multi_logloss: 0.0543957

Training time: 47.9 seconds

LightGBM Balanced Accuracy: 0.965331

Classification Report:
              precision    recall  f1-score   support

        High       0.96      0.93      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.99      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.97      0.97    126000
weighted avg       0.99      0.99      0.99    126000




## Model 2 — Probability Optimization and Ensemble

Optimize LightGBM predictions for balanced accuracy and evaluate a CatBoost + LightGBM probability ensemble.


In [9]:
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

# Align LightGBM class order with CatBoost
classes = catboost_model.classes_

lgb_aligned = lgb_probs[
    :, [list(lgb_model.classes_).index(c) for c in classes]
]

cat_probs = catboost_model.predict_proba(X_val)

results = []

# Test LightGBM alone and probability ensembles
for blend_weight in [0.0, 0.25, 0.5, 0.75, 1.0]:

    # 0 = LightGBM only, 1 = CatBoost only
    blended_probs = (
        blend_weight * cat_probs
        + (1 - blend_weight) * lgb_aligned
    )

    for high_weight in np.arange(1.0, 8.01, 0.1):

        weights = np.array([
            high_weight if c == "High" else 1.0
            for c in classes
        ])

        predictions = classes[
            np.argmax(blended_probs * weights, axis=1)
        ]

        score = balanced_accuracy_score(y_val, predictions)

        results.append({
            "catboost_weight": blend_weight,
            "lightgbm_weight": 1 - blend_weight,
            "high_multiplier": round(high_weight, 2),
            "balanced_accuracy": score
        })

results_df = pd.DataFrame(results)

print("TOP 15 CONFIGURATIONS:")
display(
    results_df.sort_values(
        "balanced_accuracy",
        ascending=False
    ).head(15)
)

print("\nBEST CONFIGURATION:")
display(
    results_df.loc[
        [results_df["balanced_accuracy"].idxmax()]
    ]
)


TOP 15 CONFIGURATIONS:


,catboost_weight,lightgbm_weight,high_multiplier,balanced_accuracy
139,0.25,0.75,7.8,0.973210
140,0.25,0.75,7.9,0.973203
141,0.25,0.75,8.0,0.973134
137,0.25,0.75,7.6,0.973049
132,0.25,0.75,7.1,0.973044
138,0.25,0.75,7.7,0.973007
136,0.25,0.75,7.5,0.972998
133,0.25,0.75,7.2,0.972995
62,0.00,1.00,7.2,0.972984
134,0.25,0.75,7.3,0.972967



BEST CONFIGURATION:


,catboost_weight,lightgbm_weight,high_multiplier,balanced_accuracy
139,0.25,0.75,7.8,0.97321


In [10]:

import numpy as np
from pathlib import Path

# Prepare test features
X_test = test.drop(columns=["id"]).copy()

# Convert categorical columns for LightGBM
X_test_lgb = X_test.copy()

for col in cat_features:
    X_test_lgb[col] = pd.Categorical(
        X_test_lgb[col],
        categories=X_train_lgb[col].cat.categories
    )

# CatBoost test probabilities
cat_test_probs = catboost_model.predict_proba(X_test)

# LightGBM test probabilities
lgb_test_probs = lgb_model.predict_proba(X_test_lgb)

# Align class order
classes = catboost_model.classes_

lgb_test_aligned = lgb_test_probs[
    :, [list(lgb_model.classes_).index(c) for c in classes]
]

# Ensemble: 25% CatBoost + 75% LightGBM
ensemble_probs = (
    0.25 * cat_test_probs
    + 0.75 * lgb_test_aligned
)

# Apply High class multiplier
weights = np.array([
    7.8 if c == "High" else 1.0
    for c in classes
])

test_predictions = classes[
    np.argmax(ensemble_probs * weights, axis=1)
]

# Create submission
submission_02 = sample_submission.copy()
submission_02["Irrigation_Need"] = test_predictions

# Validate submission
assert len(submission_02) == len(test)
assert submission_02["id"].equals(test["id"])
assert submission_02["Irrigation_Need"].isin(
    ["Low", "Medium", "High"]
).all()

# Save CSV
Path("submissions").mkdir(exist_ok=True)

submission_02.to_csv(
    "submissions/submission_02_ensemble.csv",
    index=False
)

print("Submission 2 created successfully!")
print("\nPrediction distribution:")
print(submission_02["Irrigation_Need"].value_counts())

display(submission_02.head())


Submission 2 created successfully!

Prediction distribution:
Irrigation_Need
Low       159853
Medium    100463
High        9684
Name: count, dtype: int64


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [11]:

import pandas as pd

importance = pd.DataFrame({
    "Feature": X_train_lgb.columns,
    "Importance": lgb_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance)


,Feature,Importance
7,Rainfall_mm,16447
2,Soil_Moisture,13241
5,Temperature_C,12991
9,Wind_Speed_kmh,11395
17,Previous_Irrigation_mm,11389
6,Humidity,10742
8,Sunlight_Hours,9203
1,Soil_pH,9087
15,Field_Area_hectare,8883
4,Electrical_Conductivity,8738



## Feature Engineering

Create interaction features using rainfall, soil moisture, temperature, humidity, sunlight, and previous irrigation.


In [13]:

def add_features(df):
    df = df.copy()

    df["Total_Water_Input"] = (
        df["Rainfall_mm"]
        + df["Previous_Irrigation_mm"]
    )

    df["Heat_Exposure"] = (
        df["Temperature_C"]
        * df["Sunlight_Hours"]
    )

    df["Atmospheric_Dryness"] = (
        df["Temperature_C"]
        * (1 - df["Humidity"] / 100)
    )

    df["Moisture_Rainfall_Interaction"] = (
        df["Soil_Moisture"]
        * df["Rainfall_mm"]
    )

    df["Water_Per_Hectare"] = (
        df["Total_Water_Input"]
        / (df["Field_Area_hectare"] + 0.01)
    )

    return df


# Apply feature engineering
X_train_fe = add_features(X_train_lgb)
X_val_fe = add_features(X_val_lgb)

print("Original features:", X_train_lgb.shape[1])
print("Engineered features:", X_train_fe.shape[1])

print("\nNew features:")
print(X_train_fe.columns.difference(X_train_lgb.columns).tolist())


Original features: 19
Engineered features: 24

New features:
['Atmospheric_Dryness', 'Heat_Exposure', 'Moisture_Rainfall_Interaction', 'Total_Water_Input', 'Water_Per_Hectare']



## Model 3 — LightGBM with Feature Engineering

Train LightGBM using 24 features and compare its balanced accuracy against the original LightGBM model.


In [15]:

# Remove target column if accidentally included
X_train_fe = X_train_fe.drop(
    columns=["Irrigation_Need", "id"],
    errors="ignore"
)

X_val_fe = X_val_fe.drop(
    columns=["Irrigation_Need", "id"],
    errors="ignore"
)

# Verify features
print("Training shape:", X_train_fe.shape)
print("Validation shape:", X_val_fe.shape)

print("\nTarget in features:",
      "Irrigation_Need" in X_train_fe.columns)

print("\nFeature types:")
print(X_train_fe.dtypes.value_counts())


Training shape: (504000, 24)
Validation shape: (126000, 24)

Target in features: False

Feature types:
float64     16
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64


In [18]:

import lightgbm as lgb
import time

from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report
)

# New model with engineered features
lgb_fe_model = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=40,
    colsample_bytree=0.9,
    subsample=0.8,
    subsample_freq=1,
    reg_lambda=3.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

start = time.time()


# Verify inputs before training
print("Train columns:", X_train_fe.shape[1])
print("Validation columns:", X_val_fe.shape[1])

assert "Irrigation_Need" not in X_train_fe.columns
assert "Irrigation_Need" not in X_val_fe.columns
assert X_train_fe.columns.equals(X_val_fe.columns)

# Train LightGBM
lgb_fe_model.fit(
    X_train_fe,
    y_train,
    eval_set=[(X_val_fe, y_val)],
    eval_metric="multi_logloss",
    categorical_feature=cat_features,
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(200)
    ]
)

print("Training time:",
      round(time.time() - start, 2), "seconds")

# Predict probabilities
lgb_fe_probs = lgb_fe_model.predict_proba(X_val_fe)

# Baseline predictions
lgb_fe_pred = lgb_fe_model.classes_[
    lgb_fe_probs.argmax(axis=1)
]

print("\nBalanced Accuracy:",
      round(balanced_accuracy_score(y_val, lgb_fe_pred), 6))

print("\nClassification Report:")
print(classification_report(y_val, lgb_fe_pred))


Train columns: 24
Validation columns: 24


c:\Users\Akash Kumar\Desktop\AIS-Competition-2026\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[200]	valid_0's multi_logloss: 0.0594003
[400]	valid_0's multi_logloss: 0.0562905
[600]	valid_0's multi_logloss: 0.0553659
[800]	valid_0's multi_logloss: 0.0548582
[1000]	valid_0's multi_logloss: 0.0545855
[1200]	valid_0's multi_logloss: 0.0544359
[1400]	valid_0's multi_logloss: 0.0543892
[1600]	valid_0's multi_logloss: 0.0543689
Early stopping, best iteration is:
[1571]	valid_0's multi_logloss: 0.0543499
Training time: 56.81 seconds

Balanced Accuracy: 0.965356

Classification Report:
              precision    recall  f1-score   support

        High       0.97      0.92      0.94      4202
         Low       0.99      1.00      0.99     73983
      Medium       0.99      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.97      0.97    126000
weighted avg       0.99      0.99      0.99    126000



In [19]:

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

# Align class probabilities
classes = catboost_model.classes_

fe_aligned = lgb_fe_probs[
    :, [list(lgb_fe_model.classes_).index(c) for c in classes]
]

cat_probs = catboost_model.predict_proba(X_val)

results = []

# Test different ensemble combinations
for cat_weight in [0.0, 0.1, 0.2, 0.25, 0.3, 0.4, 0.5]:

    ensemble_probs = (
        cat_weight * cat_probs
        + (1 - cat_weight) * fe_aligned
    )

    for high_weight in np.arange(5.0, 11.01, 0.2):

        weights = np.array([
            high_weight if c == "High" else 1.0
            for c in classes
        ])

        predictions = classes[
            np.argmax(ensemble_probs * weights, axis=1)
        ]

        score = balanced_accuracy_score(y_val, predictions)

        results.append({
            "catboost_weight": cat_weight,
            "lightgbm_fe_weight": 1 - cat_weight,
            "high_multiplier": round(high_weight, 2),
            "balanced_accuracy": score
        })

results_fe = pd.DataFrame(results)

print("TOP 10 CONFIGURATIONS:")
display(
    results_fe.sort_values(
        "balanced_accuracy",
        ascending=False
    ).head(10)
)


TOP 10 CONFIGURATIONS:


,catboost_weight,lightgbm_fe_weight,high_multiplier,balanced_accuracy
183,0.4,0.6,10.6,0.973413
151,0.3,0.7,10.4,0.973366
150,0.3,0.7,10.2,0.973335
154,0.3,0.7,11.0,0.973329
184,0.4,0.6,10.8,0.973329
185,0.4,0.6,11.0,0.973325
181,0.4,0.6,10.2,0.973307
92,0.2,0.8,11.0,0.973307
152,0.3,0.7,10.6,0.973303
153,0.3,0.7,10.8,0.973299


In [20]:

from sklearn.model_selection import train_test_split

# New stratified split
X_train_2, X_val_2, y_train_2, y_val_2 = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=123,
    stratify=y
)

print("New training shape:", X_train_2.shape)
print("New validation shape:", X_val_2.shape)

print("\nValidation class distribution:")
print(y_val_2.value_counts(normalize=True).round(4))


New training shape: (504000, 19)
New validation shape: (126000, 19)

Validation class distribution:
Irrigation_Need
Low       0.5872
Medium    0.3795
High      0.0333
Name: proportion, dtype: float64



## Model Validation — Second Stratified Split

Evaluate the feature-engineered LightGBM model using a different random split to assess the stability of its performance.


In [21]:

import lightgbm as lgb
import pandas as pd
import time
from sklearn.metrics import balanced_accuracy_score

# Prepare second split
X_train_2_lgb = X_train_2.copy()
X_val_2_lgb = X_val_2.copy()

# Convert categorical features
for col in cat_features:
    categories = pd.Categorical(
        X_train_2[col]
    ).categories

    X_train_2_lgb[col] = pd.Categorical(
        X_train_2[col],
        categories=categories
    )

    X_val_2_lgb[col] = pd.Categorical(
        X_val_2[col],
        categories=categories
    )

# Add engineered features
X_train_2_fe = add_features(X_train_2_lgb)
X_val_2_fe = add_features(X_val_2_lgb)

# Initialize model
lgb_fe_model_2 = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=40,
    colsample_bytree=0.9,
    subsample=0.8,
    subsample_freq=1,
    reg_lambda=3.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

start = time.time()

# Train
lgb_fe_model_2.fit(
    X_train_2_fe,
    y_train_2,
    eval_set=[(X_val_2_fe, y_val_2)],
    eval_metric="multi_logloss",
    categorical_feature=cat_features,
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(200)
    ]
)

print("\nTraining time:",
      round(time.time() - start, 2), "seconds")

# Validation probabilities
probs_2 = lgb_fe_model_2.predict_proba(X_val_2_fe)
classes_2 = lgb_fe_model_2.classes_

# Compare fixed High multipliers
for high_weight in [1.0, 7.8, 10.6]:

    weights = [
        high_weight if c == "High" else 1.0
        for c in classes_2
    ]

    predictions = classes_2[
        (probs_2 * weights).argmax(axis=1)
    ]

    score = balanced_accuracy_score(y_val_2, predictions)

    print(
        f"High multiplier {high_weight}: "
        f"Balanced Accuracy = {score:.6f}"
    )


c:\Users\Akash Kumar\Desktop\AIS-Competition-2026\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[200]	valid_0's multi_logloss: 0.0618881
[400]	valid_0's multi_logloss: 0.0586956
[600]	valid_0's multi_logloss: 0.057728
[800]	valid_0's multi_logloss: 0.057304
[1000]	valid_0's multi_logloss: 0.0570314
[1200]	valid_0's multi_logloss: 0.0569321
[1400]	valid_0's multi_logloss: 0.0568553
Early stopping, best iteration is:
[1431]	valid_0's multi_logloss: 0.0568493

Training time: 54.48 seconds
High multiplier 1.0: Balanced Accuracy = 0.961604
High multiplier 7.8: Balanced Accuracy = 0.969638
High multiplier 10.6: Balanced Accuracy = 0.970685


In [22]:

from catboost import CatBoostClassifier
from sklearn.metrics import balanced_accuracy_score
import numpy as np
import time

catboost_model_2 = CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    thread_count=-1,
    verbose=200
)

start = time.time()

catboost_model_2.fit(
    X_train_2,
    y_train_2,
    cat_features=cat_features,
    eval_set=(X_val_2, y_val_2),
    early_stopping_rounds=50
)

print("Training time:", round(time.time() - start, 2), "seconds")

# Align class probabilities
classes = catboost_model_2.classes_

cat_probs_2 = catboost_model_2.predict_proba(X_val_2)

lgb_probs_2 = lgb_fe_model_2.predict_proba(X_val_2_fe)

lgb_probs_2 = lgb_probs_2[
    :, [list(lgb_fe_model_2.classes_).index(c) for c in classes]
]

# Evaluate fixed ensemble configurations
for cat_weight, high_weight in [
    (0.25, 7.8),
    (0.40, 10.6)
]:
    ensemble = (
        cat_weight * cat_probs_2
        + (1 - cat_weight) * lgb_probs_2
    )

    weights = np.array([
        high_weight if c == "High" else 1.0
        for c in classes
    ])

    predictions = classes[
        np.argmax(ensemble * weights, axis=1)
    ]

    score = balanced_accuracy_score(y_val_2, predictions)

    print(
        f"CatBoost {cat_weight:.0%}, "
        f"LightGBM {1-cat_weight:.0%}, "
        f"High multiplier {high_weight}: "
        f"Balanced Accuracy = {score:.6f}"
    )


0:	learn: 1.0066570	test: 1.0066506	best: 1.0066506 (0)	total: 514ms	remaining: 5m 8s
200:	learn: 0.0623529	test: 0.0630049	best: 0.0630049 (200)	total: 2m 59s	remaining: 5m 55s
400:	learn: 0.0602899	test: 0.0613708	best: 0.0613708 (400)	total: 4m 17s	remaining: 2m 7s
599:	learn: 0.0588264	test: 0.0604086	best: 0.0604086 (599)	total: 5m 42s	remaining: 0us

bestTest = 0.06040857342
bestIteration = 599

Training time: 344.73 seconds
CatBoost 25%, LightGBM 75%, High multiplier 7.8: Balanced Accuracy = 0.969340
CatBoost 40%, LightGBM 60%, High multiplier 10.6: Balanced Accuracy = 0.970423


In [23]:

# Prepare full training features and target
X_full = train.drop(columns=["id", "Irrigation_Need"])
y_full = train["Irrigation_Need"]

X_test_full = test.drop(columns=["id"])

# Convert categorical columns for LightGBM
X_full_lgb = X_full.copy()
X_test_lgb = X_test_full.copy()

for col in cat_features:
    categories = pd.Categorical(X_full[col]).categories

    X_full_lgb[col] = pd.Categorical(
        X_full[col],
        categories=categories
    )

    X_test_lgb[col] = pd.Categorical(
        X_test_full[col],
        categories=categories
    )

# Feature engineering
X_full_fe = add_features(X_full_lgb)
X_test_fe = add_features(X_test_lgb)

# Verify
print("Full training:", X_full.shape)
print("Full training + features:", X_full_fe.shape)
print("Test:", X_test_full.shape)
print("Test + features:", X_test_fe.shape)

assert X_full_fe.columns.equals(X_test_fe.columns)
assert "Irrigation_Need" not in X_full_fe.columns
assert "id" not in X_full_fe.columns

print("\nData preparation successful!")


Full training: (630000, 19)
Full training + features: (630000, 24)
Test: (270000, 19)
Test + features: (270000, 24)

Data preparation successful!



## Final Model Training — Full Dataset

Train the selected LightGBM model using all available labeled data and the engineered features for the final competition submission.


In [24]:

import lightgbm as lgb
import time

# Final LightGBM model
lgb_final = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=40,
    colsample_bytree=0.9,
    subsample=0.8,
    subsample_freq=1,
    reg_lambda=3.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

start = time.time()

lgb_final.fit(
    X_full_fe,
    y_full,
    categorical_feature=cat_features
)

print("Training time:", round(time.time() - start, 2), "seconds")
print("Number of trees:", lgb_final.n_estimators_)
print("Classes:", lgb_final.classes_)

# Predict probabilities for Kaggle test data
lgb_test_probs = lgb_final.predict_proba(X_test_fe)

print("Prediction shape:", lgb_test_probs.shape)


Training time: 81.86 seconds
Number of trees: 1500
Classes: ['High' 'Low' 'Medium']
Prediction shape: (270000, 3)


In [25]:

from catboost import CatBoostClassifier
import time

# Final CatBoost model
catboost_final = CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    thread_count=-1,
    verbose=200
)

start = time.time()

# Train on all available labeled data
catboost_final.fit(
    X_full,
    y_full,
    cat_features=cat_features
)

print("\nTraining time:", round(time.time() - start, 2), "seconds")
print("Classes:", catboost_final.classes_)

# Predict test probabilities
catboost_test_probs = catboost_final.predict_proba(X_test_full)

print("Prediction shape:", catboost_test_probs.shape)


0:	learn: 1.0059036	total: 426ms	remaining: 4m 15s
200:	learn: 0.0617122	total: 1m 13s	remaining: 2m 26s
400:	learn: 0.0598219	total: 2m 28s	remaining: 1m 13s
599:	learn: 0.0586356	total: 3m 45s	remaining: 0us

Training time: 227.36 seconds
Classes: ['High' 'Low' 'Medium']
Prediction shape: (270000, 3)


In [26]:

import numpy as np
import pandas as pd
from pathlib import Path

# Ensure both models use the same class order
classes = np.asarray(catboost_final.classes_)

lgb_aligned = lgb_test_probs[
    :, [list(lgb_final.classes_).index(c) for c in classes]
]

# Ensemble probabilities
ensemble_test_probs = (
    0.40 * catboost_test_probs
    + 0.60 * lgb_aligned
)

# Apply High class multiplier
class_weights = np.array([
    10.6 if c == "High" else 1.0
    for c in classes
])

adjusted_probs = ensemble_test_probs * class_weights

# Final predictions
final_predictions = classes[
    np.argmax(adjusted_probs, axis=1)
]

# Create submission
submission_03 = sample_submission.copy()
submission_03["Irrigation_Need"] = final_predictions

# Verify submission
assert len(submission_03) == len(test)
assert submission_03["id"].equals(test["id"])
assert not submission_03.isna().any().any()

# Save
Path("submissions").mkdir(exist_ok=True)

submission_path = "submissions/submission_03_ensemble_fe.csv"
submission_03.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print("Submission shape:", submission_03.shape)

print("\nPrediction distribution:")
print(submission_03["Irrigation_Need"].value_counts())

print("\nPreview:")
print(submission_03.head())


Saved: submissions/submission_03_ensemble_fe.csv
Submission shape: (270000, 2)

Prediction distribution:
Irrigation_Need
Low       159848
Medium    100128
High       10024
Name: count, dtype: int64

Preview:
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low


In [27]:

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

classes = np.asarray(catboost_model.classes_)

# Align probability columns
cb_val = catboost_model.predict_proba(X_val)

lgb_val = lgb_probs[
    :, [list(lgb_model.classes_).index(c) for c in classes]
]

lgb_fe_val = lgb_fe_probs[
    :, [list(lgb_fe_model.classes_).index(c) for c in classes]
]

# Compare model combinations with fixed class adjustment
combinations = {
    "Submission 2": (0.25, 0.75, 0.00, 7.8),
    "Submission 3": (0.40, 0.00, 0.60, 10.6),
    "Three-model A": (0.30, 0.35, 0.35, 9.0),
    "Three-model B": (0.40, 0.30, 0.30, 10.0),
    "Three-model C": (0.20, 0.40, 0.40, 10.0),
    "Three-model D": (0.30, 0.20, 0.50, 10.0),
}

results = []

for name, (w_cb, w_lgb, w_fe, high_mult) in combinations.items():
    blended = (
        w_cb * cb_val
        + w_lgb * lgb_val
        + w_fe * lgb_fe_val
    )

    class_weights = np.array([
        high_mult if c == "High" else 1.0
        for c in classes
    ])

    preds = classes[np.argmax(blended * class_weights, axis=1)]

    results.append({
        "Model": name,
        "CatBoost": w_cb,
        "LightGBM": w_lgb,
        "LightGBM-FE": w_fe,
        "High multiplier": high_mult,
        "Balanced Accuracy": balanced_accuracy_score(y_val, preds)
    })

comparison = pd.DataFrame(results).sort_values(
    "Balanced Accuracy", ascending=False
)

display(comparison)


,Model,CatBoost,LightGBM,LightGBM-FE,High multiplier,Balanced Accuracy
5,Three-model D,0.30,0.20,0.50,10.0,0.973678
3,Three-model B,0.40,0.30,0.30,10.0,0.973559
2,Three-model A,0.30,0.35,0.35,9.0,0.973453
1,Submission 3,0.40,0.00,0.60,10.6,0.973413
4,Three-model C,0.20,0.40,0.40,10.0,0.973391
0,Submission 2,0.25,0.75,0.00,7.8,0.973210


In [28]:

import lightgbm as lgb
import numpy as np
from sklearn.metrics import balanced_accuracy_score

# Original LightGBM on second split
lgb_model_2 = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=40,
    colsample_bytree=0.9,
    subsample=0.8,
    subsample_freq=1,
    reg_lambda=3.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_model_2.fit(
    X_train_2_lgb,
    y_train_2,
    categorical_feature=cat_features
)

classes = np.asarray(catboost_model_2.classes_)

# Align predictions
cb2 = catboost_model_2.predict_proba(X_val_2)

lgb2 = lgb_model_2.predict_proba(X_val_2_lgb)
lgb2 = lgb2[:, [
    list(lgb_model_2.classes_).index(c)
    for c in classes
]]

fe2 = lgb_fe_model_2.predict_proba(X_val_2_fe)
fe2 = fe2[:, [
    list(lgb_fe_model_2.classes_).index(c)
    for c in classes
]]

# Compare both configurations
configs = {
    "Submission 3": (0.40, 0.00, 0.60, 10.6),
    "Three-model B": (0.40, 0.30, 0.30, 10.0),
    "Three-model D": (0.30, 0.20, 0.50, 10.0)
}

for name, (w_cb, w_lgb, w_fe, high_mult) in configs.items():

    blend = w_cb * cb2 + w_lgb * lgb2 + w_fe * fe2

    weights = np.array([
        high_mult if c == "High" else 1.0
        for c in classes
    ])

    predictions = classes[
        np.argmax(blend * weights, axis=1)
    ]

    score = balanced_accuracy_score(y_val_2, predictions)

    print(f"{name}: {score:.6f}")


Submission 3: 0.970423
Three-model B: 0.970432
Three-model D: 0.970579


In [29]:

import lightgbm as lgb
import time

# Full-data LightGBM without feature engineering
lgb_original_final = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=40,
    colsample_bytree=0.9,
    subsample=0.8,
    subsample_freq=1,
    reg_lambda=3.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

start = time.time()

lgb_original_final.fit(
    X_full_lgb,
    y_full,
    categorical_feature=cat_features
)

print("Training time:", round(time.time() - start, 2), "seconds")
print("Classes:", lgb_original_final.classes_)

# Predict test probabilities
lgb_original_test_probs = lgb_original_final.predict_proba(
    X_test_lgb
)

print("Prediction shape:", lgb_original_test_probs.shape)


Training time: 50.44 seconds
Classes: ['High' 'Low' 'Medium']
Prediction shape: (270000, 3)


In [30]:

import xgboost as xgb
import numpy as np
import time

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, classification_report

# Encode target labels
label_encoder = LabelEncoder()

y_train_xgb = label_encoder.fit_transform(y_train)
y_val_xgb = label_encoder.transform(y_val)

# XGBoost categorical feature support
X_train_xgb = X_train_fe.copy()
X_val_xgb = X_val_fe.copy()

for col in cat_features:
    X_train_xgb[col] = X_train_xgb[col].astype("category")
    X_val_xgb[col] = X_val_xgb[col].astype("category")

# Initialize XGBoost
xgb_model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=1800,
    max_depth=7,
    learning_rate=0.035,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=5.0,
    tree_method="hist",
    enable_categorical=True,
    eval_metric="mlogloss",
    early_stopping_rounds=100,
    random_state=42,
    n_jobs=-1
)

start = time.time()

xgb_model.fit(
    X_train_xgb,
    y_train_xgb,
    eval_set=[(X_val_xgb, y_val_xgb)],
    verbose=200
)

print("\nTraining time:", round(time.time() - start, 2), "seconds")
print("Best iteration:", xgb_model.best_iteration)

# Validation predictions
xgb_val_probs = xgb_model.predict_proba(X_val_xgb)

xgb_classes = label_encoder.classes_

# Evaluate different High class multipliers
for multiplier in [1.0, 7.8, 10.0, 12.0]:

    weights = np.array([
        multiplier if c == "High" else 1.0
        for c in xgb_classes
    ])

    predictions = xgb_classes[
        np.argmax(xgb_val_probs * weights, axis=1)
    ]

    score = balanced_accuracy_score(y_val, predictions)

    print(
        f"High multiplier {multiplier}: "
        f"Balanced Accuracy = {score:.6f}"
    )


[0]	validation_0-mlogloss:0.74952
[200]	validation_0-mlogloss:0.06000
[400]	validation_0-mlogloss:0.05639
[600]	validation_0-mlogloss:0.05536
[800]	validation_0-mlogloss:0.05494
[1000]	validation_0-mlogloss:0.05474
[1198]	validation_0-mlogloss:0.05469

Training time: 70.09 seconds
Best iteration: 1098
High multiplier 1.0: Balanced Accuracy = 0.965434
High multiplier 7.8: Balanced Accuracy = 0.972720
High multiplier 10.0: Balanced Accuracy = 0.972858
High multiplier 12.0: Balanced Accuracy = 0.973419


In [31]:

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

classes = np.asarray(catboost_model.classes_)

# Align XGBoost probabilities
xgb_aligned = xgb_val_probs[
    :, [list(xgb_classes).index(c) for c in classes]
]

# Existing validation probabilities
cb = catboost_model.predict_proba(X_val)

lgb_orig = lgb_probs[
    :, [list(lgb_model.classes_).index(c) for c in classes]
]

lgb_fe = lgb_fe_probs[
    :, [list(lgb_fe_model.classes_).index(c) for c in classes]
]

# Candidate ensembles: CB, LGB, LGB-FE, XGB, High multiplier
candidates = {
    "Three-model D": (0.30, 0.20, 0.50, 0.00, 10.0),
    "Four-model A": (0.20, 0.15, 0.35, 0.30, 10.0),
    "Four-model B": (0.20, 0.15, 0.30, 0.35, 11.0),
    "Four-model C": (0.15, 0.15, 0.30, 0.40, 12.0),
    "Four-model D": (0.25, 0.15, 0.25, 0.35, 12.0),
}

results = []

for name, (w_cb, w_lgb, w_fe, w_xgb, high_mult) in candidates.items():

    blended = (
        w_cb * cb
        + w_lgb * lgb_orig
        + w_fe * lgb_fe
        + w_xgb * xgb_aligned
    )

    weights = np.array([
        high_mult if c == "High" else 1.0
        for c in classes
    ])

    predictions = classes[
        np.argmax(blended * weights, axis=1)
    ]

    score = balanced_accuracy_score(y_val, predictions)

    results.append({
        "Model": name,
        "Balanced Accuracy": score
    })

display(
    pd.DataFrame(results).sort_values(
        "Balanced Accuracy",
        ascending=False
    )
)


,Model,Balanced Accuracy
4,Four-model D,0.973807
0,Three-model D,0.973678
2,Four-model B,0.973563
3,Four-model C,0.973542
1,Four-model A,0.973501


In [32]:

import xgboost as xgb
import numpy as np
from sklearn.metrics import balanced_accuracy_score

# Prepare second-split features
X_train_2_xgb = X_train_2_fe.copy()
X_val_2_xgb = X_val_2_fe.copy()

for col in cat_features:
    X_train_2_xgb[col] = X_train_2_xgb[col].astype("category")
    X_val_2_xgb[col] = X_val_2_xgb[col].astype("category")

# Encode labels using the existing encoder
y_train_2_xgb = label_encoder.transform(y_train_2)
y_val_2_xgb = label_encoder.transform(y_val_2)

xgb_model_2 = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=1800,
    max_depth=7,
    learning_rate=0.035,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=5.0,
    tree_method="hist",
    enable_categorical=True,
    eval_metric="mlogloss",
    early_stopping_rounds=100,
    random_state=42,
    n_jobs=-1
)

xgb_model_2.fit(
    X_train_2_xgb,
    y_train_2_xgb,
    eval_set=[(X_val_2_xgb, y_val_2_xgb)],
    verbose=200
)

# Align all validation probabilities
classes = np.asarray(catboost_model_2.classes_)

cb2 = catboost_model_2.predict_proba(X_val_2)

lgb2 = lgb_model_2.predict_proba(X_val_2_lgb)
lgb2 = lgb2[:, [
    list(lgb_model_2.classes_).index(c) for c in classes
]]

fe2 = lgb_fe_model_2.predict_proba(X_val_2_fe)
fe2 = fe2[:, [
    list(lgb_fe_model_2.classes_).index(c) for c in classes
]]

xgb2 = xgb_model_2.predict_proba(X_val_2_xgb)
xgb2 = xgb2[:, [
    list(label_encoder.classes_).index(c) for c in classes
]]

# Compare three-model and four-model ensembles
configs = {
    "Three-model D": (0.30, 0.20, 0.50, 0.00, 10.0),
    "Four-model D": (0.25, 0.15, 0.25, 0.35, 12.0)
}

for name, (w_cb, w_lgb, w_fe, w_xgb, multiplier) in configs.items():

    blended = (
        w_cb * cb2
        + w_lgb * lgb2
        + w_fe * fe2
        + w_xgb * xgb2
    )

    weights = np.array([
        multiplier if c == "High" else 1.0
        for c in classes
    ])

    predictions = classes[
        np.argmax(blended * weights, axis=1)
    ]

    score = balanced_accuracy_score(y_val_2, predictions)

    print(f"{name}: {score:.6f}")


[0]	validation_0-mlogloss:0.74978
[200]	validation_0-mlogloss:0.06228
[400]	validation_0-mlogloss:0.05863
[600]	validation_0-mlogloss:0.05764
[800]	validation_0-mlogloss:0.05726
[1000]	validation_0-mlogloss:0.05712
[1117]	validation_0-mlogloss:0.05713
Three-model D: 0.970579
Four-model D: 0.971480


In [33]:

import xgboost as xgb
import time

# Prepare full dataset for XGBoost
X_full_xgb = X_full_fe.copy()
X_test_xgb = X_test_fe.copy()

for col in cat_features:
    X_full_xgb[col] = X_full_xgb[col].astype("category")
    X_test_xgb[col] = X_test_xgb[col].astype("category")

# Encode full training labels
y_full_xgb = label_encoder.transform(y_full)

# Final XGBoost model
xgb_final = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=1100,
    max_depth=7,
    learning_rate=0.035,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=5.0,
    tree_method="hist",
    enable_categorical=True,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

start = time.time()

xgb_final.fit(
    X_full_xgb,
    y_full_xgb,
    verbose=False
)

print("Training time:", round(time.time() - start, 2), "seconds")

# Predict test probabilities
xgb_test_probs = xgb_final.predict_proba(X_test_xgb)

print("Prediction shape:", xgb_test_probs.shape)
print("Classes:", label_encoder.classes_)


Training time: 75.68 seconds
Prediction shape: (270000, 3)
Classes: ['High' 'Low' 'Medium']


In [35]:

import numpy as np
from pathlib import Path

# Use the same class order for every model
classes = np.asarray(catboost_final.classes_)

def align_probs(probs, model_classes):
    indices = [
        list(model_classes).index(c)
        for c in classes
    ]
    return probs[:, indices]

# Align probability columns
cb_final_probs = align_probs(
    catboost_test_probs, catboost_final.classes_
)

lgb_original_probs = align_probs(
    lgb_original_test_probs, lgb_original_final.classes_
)

lgb_fe_final_probs = align_probs(
    lgb_test_probs, lgb_final.classes_
)

xgb_final_probs = align_probs(
    xgb_test_probs, label_encoder.classes_
)

# Four-model ensemble
four_model_probs = (
    0.25 * cb_final_probs
    + 0.15 * lgb_original_probs
    + 0.25 * lgb_fe_final_probs
    + 0.35 * xgb_final_probs
)

# Apply class adjustment
weights = np.array([
    12.0 if c == "High" else 1.0
    for c in classes
])

final_predictions_04 = classes[
    np.argmax(four_model_probs * weights, axis=1)
]

# Create submission
submission_04 = sample_submission.copy()
submission_04["Irrigation_Need"] = final_predictions_04

# Verify
assert len(submission_04) == len(test)
assert submission_04["id"].equals(test["id"])
assert not submission_04.isna().any().any()
assert set(submission_04["Irrigation_Need"].unique()).issubset(set(classes))

# Save without overwriting previous submissions
Path("submissions").mkdir(exist_ok=True)

submission_path = "submissions/submission_04_four_model.csv"
submission_04.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print("Shape:", submission_04.shape)

print("\nPrediction distribution:")
print(submission_04["Irrigation_Need"].value_counts())

print("\nChanges compared with Submission 3:")
print(
    (submission_04["Irrigation_Need"]
     != submission_03["Irrigation_Need"]).sum()
)

print("\nPreview:")
print(submission_04 .head())


Saved: submissions/submission_04_four_model.csv
Shape: (270000, 2)

Prediction distribution:
Irrigation_Need
Low       159838
Medium     99878
High       10284
Name: count, dtype: int64

Changes compared with Submission 3:
460

Preview:
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low


In [36]:

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

# Four-model ensemble — Split 1
blend_1 = (
    0.25 * cb
    + 0.15 * lgb_orig
    + 0.25 * lgb_fe
    + 0.35 * xgb_aligned
)

# Four-model ensemble — Split 2
blend_2 = (
    0.25 * cb2
    + 0.15 * lgb2
    + 0.25 * fe2
    + 0.35 * xgb2
)

# Ensure class orders are consistent
assert list(catboost_model.classes_) == list(catboost_model_2.classes_)

classes = np.asarray(catboost_model.classes_)

results = []

for multiplier in [8, 9, 10, 11, 12, 13, 14, 15, 16]:

    weights = np.array([
        multiplier if c == "High" else 1.0
        for c in classes
    ])

    pred_1 = classes[np.argmax(blend_1 * weights, axis=1)]
    pred_2 = classes[np.argmax(blend_2 * weights, axis=1)]

    score_1 = balanced_accuracy_score(y_val, pred_1)
    score_2 = balanced_accuracy_score(y_val_2, pred_2)

    results.append({
        "High multiplier": multiplier,
        "Split 1": score_1,
        "Split 2": score_2,
        "Average": (score_1 + score_2) / 2,
        "Worst split": min(score_1, score_2)
    })

threshold_results = pd.DataFrame(results)

display(
    threshold_results.sort_values(
        "Worst split",
        ascending=False
    ).round(6)
)


,High multiplier,Split 1,Split 2,Average,Worst split
8,16,0.973971,0.971565,0.972768,0.971565
7,15,0.974004,0.971515,0.972759,0.971515
4,12,0.973807,0.971480,0.972643,0.971480
5,13,0.973564,0.971297,0.972430,0.971297
3,11,0.973353,0.971214,0.972284,0.971214
6,14,0.973694,0.971109,0.972402,0.971109
2,10,0.973359,0.970973,0.972166,0.970973
1,9,0.973008,0.970411,0.971709,0.970411
0,8,0.972981,0.969535,0.971258,0.969535


In [37]:

import numpy as np
from pathlib import Path

# Four-model probabilities from Step 37
classes = np.asarray(catboost_final.classes_)

# Updated High multiplier
weights = np.array([
    16.0 if c == "High" else 1.0
    for c in classes
])

# Final predictions
final_predictions_04 = classes[
    np.argmax(four_model_probs * weights, axis=1)
]

# Create submission 4
submission_04 = sample_submission.copy()
submission_04["Irrigation_Need"] = final_predictions_04

# Validate
assert len(submission_04) == len(test)
assert submission_04["id"].equals(test["id"])
assert not submission_04.isna().any().any()

# Save
Path("submissions").mkdir(exist_ok=True)

submission_path = "submissions/submission_04_four_model.csv"
submission_04.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print("Shape:", submission_04.shape)

print("\nPrediction distribution:")
print(submission_04["Irrigation_Need"].value_counts())

print("\nChanges compared with Submission 3:")
print(
    (submission_04["Irrigation_Need"]
     != submission_03["Irrigation_Need"]).sum()
)

print("\nPreview:")
print(submission_04.head())


Saved: submissions/submission_04_four_model.csv
Shape: (270000, 2)

Prediction distribution:
Irrigation_Need
Low       159838
Medium     99260
High       10902
Name: count, dtype: int64

Changes compared with Submission 3:
998

Preview:
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low


In [38]:

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split

# Four-model validation probabilities from Split 1
stack_features = np.hstack([
    cb,
    lgb_orig,
    lgb_fe,
    xgb_aligned
])

# Split existing validation predictions into
# meta-training and meta-testing portions
meta_train_idx, meta_test_idx = train_test_split(
    np.arange(len(y_val)),
    test_size=0.50,
    random_state=2026,
    stratify=y_val
)

X_meta_train = stack_features[meta_train_idx]
X_meta_test = stack_features[meta_test_idx]

y_meta_train = np.asarray(y_val)[meta_train_idx]
y_meta_test = np.asarray(y_val)[meta_test_idx]

# Train meta-model
meta_model = LogisticRegression(
    max_iter=2000,
    C=0.1,
    class_weight="balanced"
)

meta_model.fit(X_meta_train, y_meta_train)

# Evaluate
meta_probs = meta_model.predict_proba(X_meta_test)
meta_classes = meta_model.classes_

meta_predictions = meta_classes[
    np.argmax(meta_probs, axis=1)
]

meta_score = balanced_accuracy_score(
    y_meta_test,
    meta_predictions
)

# Compare existing four-model ensemble
baseline_probs = (
    0.25 * cb[meta_test_idx]
    + 0.15 * lgb_orig[meta_test_idx]
    + 0.25 * lgb_fe[meta_test_idx]
    + 0.35 * xgb_aligned[meta_test_idx]
)

weights = np.array([
    16.0 if c == "High" else 1.0
    for c in classes
])

baseline_predictions = classes[
    np.argmax(baseline_probs * weights, axis=1)
]

baseline_score = balanced_accuracy_score(
    y_meta_test,
    baseline_predictions
)

print("Stacking Balanced Accuracy:", round(meta_score, 6))
print("Existing Ensemble Accuracy:", round(baseline_score, 6))
print("Difference:", round(meta_score - baseline_score, 6))


Stacking Balanced Accuracy: 0.969311
Existing Ensemble Accuracy: 0.972967
Difference: -0.003656


In [39]:

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

classes = np.asarray(catboost_model.classes_)

# Four-model validation probabilities
blend_1 = (
    0.25 * cb
    + 0.15 * lgb_orig
    + 0.25 * lgb_fe
    + 0.35 * xgb_aligned
)

blend_2 = (
    0.25 * cb2
    + 0.15 * lgb2
    + 0.25 * fe2
    + 0.35 * xgb2
)

results = []

for high_mult in [12, 14, 16, 18]:
    for medium_mult in [0.90, 0.95, 1.00, 1.05, 1.10]:

        weights = np.array([
            high_mult if c == "High"
            else medium_mult if c == "Medium"
            else 1.0
            for c in classes
        ])

        pred1 = classes[np.argmax(blend_1 * weights, axis=1)]
        pred2 = classes[np.argmax(blend_2 * weights, axis=1)]

        score1 = balanced_accuracy_score(y_val, pred1)
        score2 = balanced_accuracy_score(y_val_2, pred2)

        results.append({
            "High multiplier": high_mult,
            "Medium multiplier": medium_mult,
            "Split 1": score1,
            "Split 2": score2,
            "Worst split": min(score1, score2),
            "Average": (score1 + score2) / 2
        })

calibration_results = pd.DataFrame(results)

display(
    calibration_results.sort_values(
        "Worst split",
        ascending=False
    ).head(10).round(6)
)


,High multiplier,Medium multiplier,Split 1,Split 2,Worst split,Average
19,18,1.10,0.974035,0.971766,0.971766,0.972900
11,16,0.95,0.973997,0.971760,0.971760,0.972879
18,18,1.05,0.973973,0.971727,0.971727,0.972850
17,18,1.00,0.973731,0.971688,0.971688,0.972710
13,16,1.05,0.973998,0.971587,0.971587,0.972792
5,14,0.90,0.973909,0.971566,0.971566,0.972737
12,16,1.00,0.973971,0.971565,0.971565,0.972768
16,18,0.95,0.973425,0.971539,0.971539,0.972482
10,16,0.90,0.973698,0.971537,0.971537,0.972618
6,14,0.95,0.973824,0.971507,0.971507,0.972665


In [40]:

import numpy as np
from sklearn.metrics import balanced_accuracy_score

classes = np.asarray(catboost_final.classes_)

# Candidate: High x18, Medium x1.10
new_weights = np.array([
    18.0 if c == "High"
    else 1.10 if c == "Medium"
    else 1.0
    for c in classes
])

candidate_predictions = classes[
    np.argmax(four_model_probs * new_weights, axis=1)
]

# Compare with actual Submission 4
previous_predictions = submission_04["Irrigation_Need"].to_numpy()

changed = candidate_predictions != previous_predictions

print("Total predictions:", len(candidate_predictions))
print("Changed predictions:", changed.sum())
print("Percentage changed:", round(changed.mean() * 100, 4), "%")

print("\nPrevious distribution:")
print(submission_04["Irrigation_Need"].value_counts())

print("\nCandidate distribution:")
print(
    __import__("pandas").Series(candidate_predictions).value_counts()
)


Total predictions: 270000
Changed predictions: 83
Percentage changed: 0.0307 %

Previous distribution:
Irrigation_Need
Low       159838
Medium     99260
High       10902
Name: count, dtype: int64

Candidate distribution:
Low       159811
Medium     99231
High       10958
Name: count, dtype: int64


In [41]:

import xgboost as xgb
import numpy as np
import time

from sklearn.metrics import balanced_accuracy_score

xgb_v2 = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=1600,
    max_depth=9,
    learning_rate=0.035,
    min_child_weight=8,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_lambda=10.0,
    reg_alpha=0.5,
    gamma=0.1,
    tree_method="hist",
    enable_categorical=True,
    eval_metric="mlogloss",
    early_stopping_rounds=100,
    random_state=42,
    n_jobs=-1
)

start = time.time()

xgb_v2.fit(
    X_train_xgb,
    y_train_xgb,
    eval_set=[(X_val_xgb, y_val_xgb)],
    verbose=200
)

print("\nTraining time:", round(time.time() - start, 2), "seconds")
print("Best iteration:", xgb_v2.best_iteration)

# Validation probabilities
xgb_v2_probs = xgb_v2.predict_proba(X_val_xgb)

classes_v2 = label_encoder.classes_

# Evaluate the alternative model
for multiplier in [10, 12, 14, 16]:

    weights = np.array([
        multiplier if c == "High" else 1.0
        for c in classes_v2
    ])

    predictions = classes_v2[
        np.argmax(xgb_v2_probs * weights, axis=1)
    ]

    score = balanced_accuracy_score(y_val, predictions)

    print(
        f"High x{multiplier}: "
        f"Balanced Accuracy = {score:.6f}"
    )


[0]	validation_0-mlogloss:0.74985
[200]	validation_0-mlogloss:0.06002
[400]	validation_0-mlogloss:0.05642
[600]	validation_0-mlogloss:0.05557
[800]	validation_0-mlogloss:0.05544
[843]	validation_0-mlogloss:0.05545

Training time: 56.44 seconds
Best iteration: 743
High x10: Balanced Accuracy = 0.972859
High x12: Balanced Accuracy = 0.972585
High x14: Balanced Accuracy = 0.972593
High x16: Balanced Accuracy = 0.972813


In [42]:

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

classes = np.asarray(catboost_model.classes_)

# Align XGBoost V2 probabilities with CatBoost class order
v2_class_indices = [
    np.where(label_encoder.classes_ == c)[0][0]
    for c in classes
]

xgb_v2_aligned = xgb_v2_probs[:, v2_class_indices]

# Original four-model ensemble
original_blend = (
    0.25 * cb
    + 0.15 * lgb_orig
    + 0.25 * lgb_fe
    + 0.35 * xgb_aligned
)

# New ensemble with XGBoost V2
v2_blend = (
    0.25 * cb
    + 0.15 * lgb_orig
    + 0.25 * lgb_fe
    + 0.35 * xgb_v2_aligned
)

results = []

for high_mult in [10, 12, 14, 16, 18]:
    for medium_mult in [1.0, 1.05, 1.10]:

        weights = np.array([
            high_mult if c == "High"
            else medium_mult if c == "Medium"
            else 1.0
            for c in classes
        ])

        old_pred = classes[
            np.argmax(original_blend * weights, axis=1)
        ]

        new_pred = classes[
            np.argmax(v2_blend * weights, axis=1)
        ]

        results.append({
            "High": high_mult,
            "Medium": medium_mult,
            "Original": balanced_accuracy_score(y_val, old_pred),
            "With V2": balanced_accuracy_score(y_val, new_pred),
            "Improvement": (
                balanced_accuracy_score(y_val, new_pred)
                - balanced_accuracy_score(y_val, old_pred)
            )
        })

comparison = pd.DataFrame(results)

display(
    comparison.sort_values(
        "With V2",
        ascending=False
    ).head(10).round(6)
)


,High,Medium,Original,With V2,Improvement
3,12,1.00,0.973807,0.973777,-0.000030
14,18,1.10,0.974035,0.973737,-0.000298
4,12,1.05,0.973506,0.973600,0.000094
12,18,1.00,0.973731,0.973589,-0.000142
8,14,1.10,0.973634,0.973581,-0.000053
7,14,1.05,0.973528,0.973568,0.000040
13,18,1.05,0.973973,0.973544,-0.000429
9,16,1.00,0.973971,0.973539,-0.000432
6,14,1.00,0.973694,0.973503,-0.000191
0,10,1.00,0.973359,0.973440,0.000081


In [43]:

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

results = []

# Blend the original XGBoost and XGBoost V2
for v2_share in [0.0, 0.25, 0.50, 0.75, 1.0]:

    combined_xgb = (
        (1 - v2_share) * xgb_aligned
        + v2_share * xgb_v2_aligned
    )

    ensemble_probs = (
        0.25 * cb
        + 0.15 * lgb_orig
        + 0.25 * lgb_fe
        + 0.35 * combined_xgb
    )

    for high_mult in [12, 16, 18]:
        for medium_mult in [1.0, 1.10]:

            weights = np.array([
                high_mult if c == "High"
                else medium_mult if c == "Medium"
                else 1.0
                for c in classes
            ])

            predictions = classes[
                np.argmax(ensemble_probs * weights, axis=1)
            ]

            score = balanced_accuracy_score(y_val, predictions)

            results.append({
                "V2 share": v2_share,
                "High": high_mult,
                "Medium": medium_mult,
                "Balanced Accuracy": score
            })

blend_results = pd.DataFrame(results)

display(
    blend_results.sort_values(
        "Balanced Accuracy",
        ascending=False
    ).head(10).round(6)
)


,V2 share,High,Medium,Balanced Accuracy
11,0.25,18,1.1,0.974089
5,0.00,18,1.1,0.974035
22,0.75,18,1.0,0.973981
2,0.00,16,1.0,0.973971
8,0.25,16,1.0,0.973968
14,0.50,16,1.0,0.973936
16,0.50,18,1.0,0.973920
17,0.50,18,1.1,0.973900
20,0.75,16,1.0,0.973892
23,0.75,18,1.1,0.973863


In [44]:

from sklearn.base import clone
from sklearn.metrics import balanced_accuracy_score
import numpy as np
import time

# Train V2 on Split 2
xgb_v2_split2 = clone(xgb_v2)

start = time.time()

xgb_v2_split2.fit(
    X_train_2_fe,
    label_encoder.transform(y_train_2),
    eval_set=[
        (X_val_2_fe, label_encoder.transform(y_val_2))
    ],
    verbose=False
)

print("Training time:", round(time.time() - start, 2), "seconds")
print("Best iteration:", xgb_v2_split2.best_iteration)

# Align probabilities
v2_probs_2 = xgb_v2_split2.predict_proba(X_val_2_fe)
v2_probs_2 = v2_probs_2[:, v2_class_indices]

# 75% original XGBoost + 25% V2
combined_xgb_2 = (
    0.75 * xgb2
    + 0.25 * v2_probs_2
)

new_blend_2 = (
    0.25 * cb2
    + 0.15 * lgb2
    + 0.25 * fe2
    + 0.35 * combined_xgb_2
)

old_blend_2 = (
    0.25 * cb2
    + 0.15 * lgb2
    + 0.25 * fe2
    + 0.35 * xgb2
)

weights = np.array([
    18.0 if c == "High"
    else 1.10 if c == "Medium"
    else 1.0
    for c in classes
])

old_predictions = classes[np.argmax(old_blend_2 * weights, axis=1)]
new_predictions = classes[np.argmax(new_blend_2 * weights, axis=1)]

old_score = balanced_accuracy_score(y_val_2, old_predictions)
new_score = balanced_accuracy_score(y_val_2, new_predictions)

print(f"Original ensemble: {old_score:.6f}")
print(f"New ensemble:      {new_score:.6f}")
print(f"Improvement:       {new_score - old_score:+.6f}")
print("Changed predictions:", np.sum(old_predictions != new_predictions))


Training time: 52.49 seconds
Best iteration: 711
Original ensemble: 0.971766
New ensemble:      0.971752
Improvement:       -0.000014
Changed predictions: 24


In [45]:

import time
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score

# Use the original training and validation split
categorical_cols = X_train.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()

numerical_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_cols
        ),
        ("numerical", "passthrough", numerical_cols)
    ]
)

extra_trees_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", ExtraTreesClassifier(
        n_estimators=400,
        max_features=0.9,
        min_samples_leaf=2,
        max_depth=None,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

start = time.time()

extra_trees_model.fit(X_train, y_train)

print("Training time:", round(time.time() - start, 2), "seconds")

extra_trees_probs = extra_trees_model.predict_proba(X_val)
extra_trees_classes = extra_trees_model.classes_

for high_mult in [1, 2, 4, 6, 8, 10]:
    weights = np.array([
        high_mult if c == "High" else 1.0
        for c in extra_trees_classes
    ])

    predictions = extra_trees_classes[
        np.argmax(extra_trees_probs * weights, axis=1)
    ]

    score = balanced_accuracy_score(y_val, predictions)

    print(
        f"High x{high_mult}: "
        f"Balanced Accuracy = {score:.6f}"
    )


Training time: 225.07 seconds
High x1: Balanced Accuracy = 0.964997
High x2: Balanced Accuracy = 0.966523
High x4: Balanced Accuracy = 0.967961
High x6: Balanced Accuracy = 0.968063
High x8: Balanced Accuracy = 0.967380
High x10: Balanced Accuracy = 0.966271


In [46]:

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

# Align Extra Trees probabilities with ensemble classes
et_indices = [
    np.where(extra_trees_classes == c)[0][0]
    for c in classes
]

et_probs = extra_trees_probs[:, et_indices]

results = []

for et_weight in [0.0, 0.05, 0.10, 0.15]:

    # Preserve relative weights of existing ensemble
    combined_probs = (
        (1 - et_weight) * blend_1
        + et_weight * et_probs
    )

    for high_mult in [12, 16, 18]:
        for medium_mult in [1.0, 1.10]:

            weights = np.array([
                high_mult if c == "High"
                else medium_mult if c == "Medium"
                else 1.0
                for c in classes
            ])

            predictions = classes[
                np.argmax(combined_probs * weights, axis=1)
            ]

            score = balanced_accuracy_score(
                y_val,
                predictions
            )

            results.append({
                "Extra Trees Weight": et_weight,
                "High Multiplier": high_mult,
                "Medium Multiplier": medium_mult,
                "Balanced Accuracy": score
            })

et_results = pd.DataFrame(results)

display(
    et_results.sort_values(
        "Balanced Accuracy",
        ascending=False
    ).head(10).round(6)
)


,Extra Trees Weight,High Multiplier,Medium Multiplier,Balanced Accuracy
5,0.00,18,1.1,0.974035
2,0.00,16,1.0,0.973971
8,0.05,16,1.0,0.973948
11,0.05,18,1.1,0.973849
0,0.00,12,1.0,0.973807
14,0.10,16,1.0,0.973788
3,0.00,16,1.1,0.973759
4,0.00,18,1.0,0.973731
17,0.10,18,1.1,0.973720
7,0.05,12,1.1,0.973683


In [47]:

from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

weights = np.array([
    18.0 if c == "High"
    else 1.10 if c == "Medium"
    else 1.0
    for c in classes
])

predictions = classes[
    np.argmax(blend_2 * weights, axis=1)
]

print("Classification Report — Split 2")
print(classification_report(y_val_2, predictions, digits=5))

cm = confusion_matrix(y_val_2, predictions, labels=classes)

display(
    pd.DataFrame(
        cm,
        index=[f"Actual {c}" for c in classes],
        columns=[f"Predicted {c}" for c in classes]
    )
)


Classification Report — Split 2
              precision    recall  f1-score   support

        High    0.81629   0.96121   0.88284      4202
         Low    0.98590   0.99508   0.99047     73983
      Medium    0.98868   0.95901   0.97362     47815

    accuracy                        0.98026    126000
   macro avg    0.93029   0.97177   0.94898    126000
weighted avg    0.98130   0.98026   0.98048    126000



,Predicted High,Predicted Low,Predicted Medium
Actual High,4039,0,163
Actual Low,2,73619,362
Actual Medium,907,1053,45855


In [48]:

import pandas as pd
import numpy as np

# Existing Split 2 ensemble
weights = np.array([
    18.0 if c == "High"
    else 1.10 if c == "Medium"
    else 1.0
    for c in classes
])

weighted_probs = blend_2 * weights
normalized_probs = weighted_probs / weighted_probs.sum(axis=1, keepdims=True)

predictions = classes[np.argmax(weighted_probs, axis=1)]
actual = np.asarray(y_val_2)

confidence = normalized_probs.max(axis=1)

errors = pd.DataFrame({
    "Actual": actual,
    "Predicted": predictions,
    "Confidence": confidence
})

errors = errors[errors["Actual"] != errors["Predicted"]]

print("Total mistakes:", len(errors))

print("\nMistakes by class pair:")
display(
    errors.groupby(["Actual", "Predicted"])
    .agg(
        Count=("Confidence", "size"),
        Mean_Confidence=("Confidence", "mean"),
        Median_Confidence=("Confidence", "median")
    )
    .round(4)
)

print("\nConfidence distribution of mistakes:")
display(errors["Confidence"].describe().round(4))


Total mistakes: 2487

Mistakes by class pair:


Count  Mean_Confidence  Median_Confidence
Actual Predicted                                           
High   Medium       163           0.7362             0.7249
Low    High           2           0.4911             0.4911
       Medium       362           0.8658             0.9428
Medium High         907           0.7111             0.6782
       Low         1053           0.8971             0.9315


Confidence distribution of mistakes:


count    2487.0000
mean        0.8138
std         0.1596
min         0.3932
25%         0.6736
50%         0.8715
75%         0.9519
max         0.9998
Name: Confidence, dtype: float64

In [49]:

from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, balanced_accuracy_score
import numpy as np
import time

# Train only on Low and Medium samples
binary_mask = y_train.isin(["Low", "Medium"])

X_binary_train = X_train.loc[binary_mask]
y_binary_train = y_train.loc[binary_mask]

# Validation: only Low and Medium samples
binary_val_mask = y_val.isin(["Low", "Medium"])

X_binary_val = X_val.loc[binary_val_mask]
y_binary_val = y_val.loc[binary_val_mask]

medium_low_model = CatBoostClassifier(
    iterations=900,
    depth=8,
    learning_rate=0.04,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=5,
    random_seed=42,
    thread_count=-1,
    verbose=200
)

start = time.time()

medium_low_model.fit(
    X_binary_train,
    y_binary_train,
    cat_features=cat_features,
    eval_set=(X_binary_val, y_binary_val),
    early_stopping_rounds=100
)

print("\nTraining time:", round(time.time() - start, 2), "seconds")
print("Best iteration:", medium_low_model.best_iteration_)

binary_predictions = medium_low_model.predict(
    X_binary_val
).ravel()

print("\nMedium vs Low classification report:")
print(
    classification_report(
        y_binary_val,
        binary_predictions,
        digits=5
    )
)

print(
    "Binary Balanced Accuracy:",
    balanced_accuracy_score(
        y_binary_val,
        binary_predictions
    )
)


0:	test: 0.9968251	best: 0.9968251 (0)	total: 313ms	remaining: 4m 41s
200:	test: 0.9974378	best: 0.9974378 (200)	total: 46.5s	remaining: 2m 41s
400:	test: 0.9975422	best: 0.9975422 (400)	total: 1m 36s	remaining: 2m
600:	test: 0.9976050	best: 0.9976050 (597)	total: 2m 27s	remaining: 1m 13s
800:	test: 0.9976417	best: 0.9976417 (799)	total: 3m 18s	remaining: 24.5s
899:	test: 0.9976631	best: 0.9976635 (895)	total: 3m 43s	remaining: 0us

bestTest = 0.9976635283
bestIteration = 895

Shrink model to first 896 iterations.

Training time: 225.14 seconds
Best iteration: 895

Medium vs Low classification report:
              precision    recall  f1-score   support

         Low    0.98656   0.99489   0.99071     73983
      Medium    0.99199   0.97902   0.98546     47815

    accuracy                        0.98866    121798
   macro avg    0.98927   0.98696   0.98808    121798
weighted avg    0.98869   0.98866   0.98865    121798

Binary Balanced Accuracy: 0.986957018590046


In [50]:

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, classification_report

# Original four-model ensemble
weights = np.array([
    18.0 if c == "High"
    else 1.10 if c == "Medium"
    else 1.0
    for c in classes
])

original_predictions = classes[
    np.argmax(blend_1 * weights, axis=1)
]

# Specialist probabilities on all validation samples
specialist_probs = medium_low_model.predict_proba(X_val)

specialist_classes = np.asarray(medium_low_model.classes_)

medium_idx = np.where(specialist_classes == "Medium")[0][0]
low_idx = np.where(specialist_classes == "Low")[0][0]

# Original ensemble's weighted Medium/Low probabilities
ensemble_medium_idx = np.where(classes == "Medium")[0][0]
ensemble_low_idx = np.where(classes == "Low")[0][0]

weighted_ensemble = blend_1 * weights

medium_low_total = (
    weighted_ensemble[:, ensemble_medium_idx]
    + weighted_ensemble[:, ensemble_low_idx]
)

original_medium_prob = (
    weighted_ensemble[:, ensemble_medium_idx]
    / medium_low_total
)

# Compare different strengths of specialist blending
results = []

for specialist_weight in [0.0, 0.25, 0.50, 0.75, 1.0]:

    combined_medium_prob = (
        (1 - specialist_weight) * original_medium_prob
        + specialist_weight * specialist_probs[:, medium_idx]
    )

    new_predictions = original_predictions.copy()

    # Only change Low/Medium predictions, never High
    eligible = original_predictions != "High"

    new_predictions[eligible] = np.where(
        combined_medium_prob[eligible] >= 0.5,
        "Medium",
        "Low"
    )

    score = balanced_accuracy_score(y_val, new_predictions)

    results.append({
        "Specialist Weight": specialist_weight,
        "Balanced Accuracy": score,
        "Changed Predictions": np.sum(
            new_predictions != original_predictions
        )
    })

display(
    pd.DataFrame(results).round(6)
)

print(
    "\nOriginal Balanced Accuracy:",
    balanced_accuracy_score(y_val, original_predictions)
)


,Specialist Weight,Balanced Accuracy,Changed Predictions
0,0.00,0.974035,0
1,0.25,0.974019,27
2,0.50,0.974000,64
3,0.75,0.973920,95
4,1.00,0.973825,117



Original Balanced Accuracy: 0.9740348658477722


In [51]:

import pandas as pd
import numpy as np

weights = np.array([
    18.0 if c == "High"
    else 1.10 if c == "Medium"
    else 1.0
    for c in classes
])

pred = classes[np.argmax(blend_2 * weights, axis=1)]

analysis_df = X_val_2.copy()
analysis_df["Actual"] = np.asarray(y_val_2)
analysis_df["Predicted"] = pred
analysis_df["Is_Error"] = (
    analysis_df["Actual"] != analysis_df["Predicted"]
)

for col in ["Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Irrigation_Type"]:

    summary = (
        analysis_df.groupby(col, observed=True)
        .agg(
            Samples=("Is_Error", "size"),
            Errors=("Is_Error", "sum"),
            Error_Rate=("Is_Error", "mean")
        )
        .sort_values("Error_Rate", ascending=False)
    )

    summary["Error_Rate"] *= 100

    print(f"\n{col}")
    display(summary.round(3))



Soil_Type


,Samples,Errors,Error_Rate
Soil_Type,,,
Sandy,33316,695,2.086
Clay,31825,640,2.011
Silt,29653,587,1.980
Loamy,31206,565,1.811



Crop_Type


,Samples,Errors,Error_Rate
Crop_Type,,,
Maize,20804,438,2.105
Sugarcane,21814,450,2.063
Cotton,20768,424,2.042
Wheat,20783,417,2.006
Potato,20295,374,1.843
Rice,21536,384,1.783



Crop_Growth_Stage


,Samples,Errors,Error_Rate
Crop_Growth_Stage,,,
Vegetative,31415,866,2.757
Flowering,31559,838,2.655
Harvest,33571,448,1.334
Sowing,29455,335,1.137



Irrigation_Type


,Samples,Errors,Error_Rate
Irrigation_Type,,,
Canal,32253,678,2.102
Sprinkler,32339,643,1.988
Rainfed,31258,595,1.904
Drip,30150,571,1.894


In [52]:

import pandas as pd
import numpy as np

# Reuse Split 2 predictions from Step 53
stage_analysis = pd.DataFrame({
    "Growth_Stage": X_val_2["Crop_Growth_Stage"].to_numpy(),
    "Actual": np.asarray(y_val_2),
    "Predicted": pred
})

stage_analysis["Correct"] = (
    stage_analysis["Actual"] == stage_analysis["Predicted"]
)

stage_recall = (
    stage_analysis
    .groupby(["Growth_Stage", "Actual"])
    .agg(
        Samples=("Correct", "size"),
        Correct=("Correct", "sum"),
        Recall=("Correct", "mean")
    )
    .reset_index()
)

stage_recall["Recall"] *= 100

display(
    stage_recall.sort_values(
        ["Growth_Stage", "Actual"]
    ).round(3)
)


,Growth_Stage,Actual,Samples,Correct,Recall
0,Flowering,High,2028,1957,96.499
1,Flowering,Low,9782,9702,99.182
2,Flowering,Medium,19749,19062,96.521
3,Harvest,High,109,101,92.661
4,Harvest,Low,28436,28316,99.578
5,Harvest,Medium,5026,4706,93.633
6,Sowing,High,43,40,93.023
7,Sowing,Low,25662,25586,99.704
8,Sowing,Medium,3750,3494,93.173
9,Vegetative,High,2022,1941,95.994


In [53]:

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

def stage_adjusted_predictions(probs, stages, high_adjustments):
    weighted = probs.copy()

    high_idx = np.where(classes == "High")[0][0]
    medium_idx = np.where(classes == "Medium")[0][0]

    weighted[:, high_idx] *= 18.0
    weighted[:, medium_idx] *= 1.10

    stages = np.asarray(stages)

    for stage, adjustment in high_adjustments.items():
        mask = stages == stage
        weighted[mask, high_idx] *= adjustment

    return classes[np.argmax(weighted, axis=1)]


results = []

# Small, predefined set of stage-specific adjustments
for adjustment in [0.90, 0.95, 1.00, 1.05, 1.10]:

    adjustments = {
        "Sowing": adjustment,
        "Harvest": adjustment
    }

    pred1 = stage_adjusted_predictions(
        blend_1,
        X_val["Crop_Growth_Stage"],
        adjustments
    )

    pred2 = stage_adjusted_predictions(
        blend_2,
        X_val_2["Crop_Growth_Stage"],
        adjustments
    )

    results.append({
        "Adjustment": adjustment,
        "Split 1": balanced_accuracy_score(y_val, pred1),
        "Split 2": balanced_accuracy_score(y_val_2, pred2)
    })

display(pd.DataFrame(results).round(6))


,Adjustment,Split 1,Split 2
0,0.90,0.974098,0.971794
1,0.95,0.974063,0.971766
2,1.00,0.974035,0.971766
3,1.05,0.974014,0.971745
4,1.10,0.973993,0.971738


In [54]:

import numpy as np
import pandas as pd

# Baseline weights: High x18, Medium x1.10
baseline_weights = np.array([
    18.0 if c == "High"
    else 1.10 if c == "Medium"
    else 1.0
    for c in classes
])

baseline_test = classes[
    np.argmax(four_model_probs * baseline_weights, axis=1)
]

# Apply stage-specific adjustment
adjusted_probs = four_model_probs.copy() * baseline_weights

high_idx = np.where(classes == "High")[0][0]

stage_mask = X_test_full["Crop_Growth_Stage"].isin(
    ["Sowing", "Harvest"]
).to_numpy()

adjusted_probs[stage_mask, high_idx] *= 0.90

adjusted_test = classes[
    np.argmax(adjusted_probs, axis=1)
]

changed = adjusted_test != baseline_test

print("Total test predictions:", len(adjusted_test))
print("Changed predictions:", changed.sum())
print("Percentage changed:", round(changed.mean() * 100, 4), "%")

print("\nChanges by growth stage:")
display(
    pd.crosstab(
        X_test_full.loc[changed, "Crop_Growth_Stage"],
        [
            pd.Series(baseline_test[changed], index=X_test_full.index[changed]),
            pd.Series(adjusted_test[changed], index=X_test_full.index[changed])
        ],
        rownames=["Growth Stage"],
        colnames=["Before", "After"]
    )
)


Total test predictions: 270000
Changed predictions: 16
Percentage changed: 0.0059 %

Changes by growth stage:


Before,High
After,Medium
Growth Stage,
Harvest,12
Sowing,4
